# LeetCode 718 — Maximum Length of Repeated Subarray bằng Python

Link đề: https://leetcode.com/problems/maximum-length-of-repeated-subarray/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Dynamic Programming, Sliding Window, Hash Function, Rolling Hash

Skeleton LeetCode:

```python
class Solution:
    def findLength(self, nums1: List[int], nums2: List[int]) -> int:
        ...
```

Ghi chú: các ví dụ ngoài hai ví dụ của đề (ví dụ 3 và 4 ở mục 2) do tôi tự đặt, đáp án đã kiểm chứng bằng vét cạn; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **độ dài lớn nhất** của một **mảng con liên tiếp (subarray)** xuất hiện ở **cả hai** mảng `nums1` và `nums2`; phân biệt rõ **subarray (liên tiếp)** với **subsequence (dãy con, có thể nhảy cóc)**.
2. Nắm các công cụ Python: tạo bảng 2 chiều `[[0] * (n + 1) for _ in range(m + 1)]`, cắt lát `a[i:i + L]`, `tuple` và `set`, tìm kiếm nhị phân trên **đáp án**, băm cuộn (**rolling hash**) với phép `% mod`, và `max` kèm giá trị mặc định.
3. Đi từ **thử mọi cặp vị trí bắt đầu rồi mở rộng** → **quy hoạch động 2 chiều (hậu tố chung dài nhất)** → **rút gọn thành 1 hàng** → **tìm kiếm nhị phân độ dài + băm cuộn** → **trượt hai mảng lệch nhau và đếm đoạn khớp dài nhất trên mỗi đường chéo**.
4. Hiểu mấu chốt: gọi `f[i][j]` là độ dài **đoạn chung dài nhất kết thúc đúng tại** `nums1[i-1]` và `nums2[j-1]`; nếu hai phần tử bằng nhau thì `f[i][j] = f[i-1][j-1] + 1`, ngược lại `f[i][j] = 0` (đoạn bị **đứt**); đáp án là **max của cả bảng**.
5. Tránh các lỗi kinh điển: dùng công thức LCS (cho dãy con) thay vì "đứt thì về `0`", duyệt `j` tăng dần khi rút gọn một hàng, lấy `f[m][n]` thay vì `max`, bảng 2 chiều bị **dùng chung hàng** (`[[0]*n]*m`), tìm kiếm nhị phân bắt đầu từ `1` làm sai khi đáp án là `0`, và tin băm mà không **kiểm chứng** lại.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `nums1` và `nums2`. Trả về **độ dài lớn nhất của một mảng con (liên tiếp) xuất hiện trong cả hai mảng**.

**Ví dụ 1** (của đề)

- Input: `nums1 = [1,2,3,2,1]`, `nums2 = [3,2,1,4,7]`
- Output: `3`
- Giải thích: mảng con lặp dài nhất là `[3,2,1]`.

**Ví dụ 2** (của đề)

- Input: `nums1 = [0,0,0,0,0]`, `nums2 = [0,0,0,0,0]`
- Output: `5`
- Giải thích: mảng con lặp dài nhất là `[0,0,0,0,0]`.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `nums1 = [1,2,3]`, `nums2 = [1,3]`
- Output: `1`
- Giải thích: `[1]` hoặc `[3]` đều là mảng con chung có độ dài `1`. Dãy con chung `[1,3]` dài `2` **không** tính vì trong `nums1` hai số `1` và `3` **không liên tiếp**.

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `nums1 = [1]`, `nums2 = [2]`
- Output: `0`
- Giải thích: không có phần tử chung nên không có mảng con chung nào (độ dài `0`).

**Ràng buộc:**

- `1 <= nums1.length, nums2.length <= 1000`
- `0 <= nums1[i], nums2[i] <= 100`

In [1]:
from typing import List
import random
import time

examples = [
    ([1, 2, 3, 2, 1], [3, 2, 1, 4, 7], 3),
    ([0, 0, 0, 0, 0], [0, 0, 0, 0, 0], 5),
    ([1, 2, 3], [1, 3], 1),
    ([1], [2], 0),
]
for a, b, expected in examples:
    print(a, b, "-> kỳ vọng", expected)

[1, 2, 3, 2, 1] [3, 2, 1, 4, 7] -> kỳ vọng 3
[0, 0, 0, 0, 0] [0, 0, 0, 0, 0] -> kỳ vọng 5
[1, 2, 3] [1, 3] -> kỳ vọng 1
[1] [2] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Mảng con (subarray) khác dãy con (subsequence)

- **Mảng con (subarray / substring)**: các phần tử **liên tiếp**, tức là một đoạn `a[i:j]`.
- **Dãy con (subsequence)**: lấy các phần tử theo thứ tự nhưng **được bỏ qua** một số phần tử ở giữa.

Với `a = [1, 2, 3]`: `[1, 2]`, `[2, 3]` là mảng con; `[1, 3]` là dãy con nhưng **không** là mảng con. Bài này hỏi về **mảng con**, nên một "vết đứt" (hai số không kề nhau) làm hỏng đoạn khớp.

In [2]:
a = [1, 2, 3]
subarrays = [a[i:j] for i in range(len(a)) for j in range(i + 1, len(a) + 1)]
print("mảng con :", subarrays)                       # 6 mảng con
print("[1,3] là mảng con của a?", [1, 3] in subarrays)   # False

mảng con : [[1], [1, 2], [1, 2, 3], [2], [2, 3], [3]]
[1,3] là mảng con của a? False


### 3.2. Cắt lát `a[i:i+L]`, `tuple`, `set`

`a[i:i + L]` là mảng con độ dài `L` bắt đầu ở `i` (tự cắt ngắn nếu vượt biên). Muốn đưa mảng con vào `set` (để tra cứu nhanh) phải đổi sang **`tuple`** vì `list` không băm được (hashable). Hai mảng con bằng nhau khi các tuple bằng nhau.

In [3]:
a = [1, 2, 3, 2, 1]
L = 3
windows = [tuple(a[i:i + L]) for i in range(len(a) - L + 1)]
print(windows)                                       # [(1,2,3), (2,3,2), (3,2,1)]
print((3, 2, 1) in set(windows))                     # True
try:
    {[1, 2]}
except TypeError as e:
    print("TypeError:", e)                           # list không băm được

[(1, 2, 3), (2, 3, 2), (3, 2, 1)]
True
TypeError: unhashable type: 'list'


### 3.3. Bảng 2 chiều và cái bẫy "dùng chung hàng"

`[[0] * (n + 1) for _ in range(m + 1)]` tạo `m + 1` hàng **độc lập**, mỗi hàng `n + 1` số `0`. Nếu viết `[[0] * (n + 1)] * (m + 1)` thì cả `m + 1` hàng là **cùng một đối tượng** list: đổi một ô là đổi mọi hàng.

In [4]:
good = [[0] * 3 for _ in range(2)]
bad = [[0] * 3] * 2
good[0][1] = 7
bad[0][1] = 7
print("good:", good)                                 # [[0, 7, 0], [0, 0, 0]]
print("bad :", bad)                                  # [[0, 7, 0], [0, 7, 0]]  <- cả hai hàng cùng đổi!
print("bad[0] is bad[1]:", bad[0] is bad[1])

good: [[0, 7, 0], [0, 0, 0]]
bad : [[0, 7, 0], [0, 7, 0]]
bad[0] is bad[1]: True


### 3.4. Tìm kiếm nhị phân **trên đáp án**

Khi bài hỏi "lớn nhất `L` sao cho điều kiện `P(L)` đúng" và `P` **đơn điệu** (đúng với `L` thì đúng với mọi `L' < L`), ta có thể **chặt nhị phân trên `L`**: mỗi lần thử một `L = mid`, nếu `P(mid)` đúng thì đáp án `≥ mid`, ngược lại `< mid`. Ở bài này `P(L)` = "có một mảng con độ dài `L` chung của hai mảng". Nếu có đoạn chung dài `L` thì **cắt bớt một đầu** ta được đoạn chung dài `L - 1`, nên `P` đơn điệu.

In [5]:
def max_true(lo, hi, P):
    """Lớn nhất L trong [lo, hi] mà P(L) đúng (P đơn điệu: đúng rồi sai); trả lo nếu P(lo) sai thì cần lo là mặc định."""
    while lo < hi:
        mid = (lo + hi + 1) // 2           # làm tròn lên để không lặp vô hạn
        if P(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

print(max_true(0, 100, lambda L: L <= 37))         # 37
print(max_true(0, 100, lambda L: L <= 0))          # 0

37
0


### 3.5. Băm cuộn (rolling hash)

Để so sánh nhanh hai mảng con độ dài `L`, ta gán mỗi mảng con một **số băm** `H = (x0·B^(L-1) + x1·B^(L-2) + ... + x(L-1)) mod M`. Hai mảng con bằng nhau thì có **cùng** `H`. Nhờ mảng băm tiền tố `P[i]` (băm của `a[:i]`) và `pw[L] = B^L mod M`, băm của `a[i:i+L]` tính trong `O(1)`:

```
H(i, L) = (P[i + L] - P[i] * pw[L]) mod M
```

Điểm cần nhớ: băm **trùng chưa chắc** mảng bằng nhau (va chạm/collision), nên khi các băm trùng ta phải **so lại thật** (`a[i:i+L] == b[j:j+L]`) hoặc chọn `M` rất lớn.

In [6]:
MOD = (1 << 61) - 1
BASE = 131


def prefix_hashes(a, base=BASE, mod=MOD):
    P = [0]
    for x in a:
        P.append((P[-1] * base + x + 1) % mod)         # +1 để giá trị 0 vẫn "có mặt"
    return P


def window_hash(P, pw, i, L, mod=MOD):
    return (P[i + L] - P[i] * pw[L]) % mod


a = [1, 2, 3, 2, 1]
P = prefix_hashes(a)
pw = [1]
for _ in range(len(a)):
    pw.append(pw[-1] * BASE % MOD)
print(window_hash(P, pw, 1, 2) == window_hash(P, pw, 1, 2))        # True
b = [9, 2, 3]
Q = prefix_hashes(b)
print(window_hash(P, pw, 1, 2) == window_hash(Q, pw, 1, 2))        # [2,3] ở cả hai -> True
print(window_hash(P, pw, 0, 2) == window_hash(Q, pw, 0, 2))        # [1,2] vs [9,2] -> False

True
True
False


### 3.6. Căn chỉnh hai mảng lệch nhau (đường chéo)

Đặt `nums1[i]` đối diện `nums2[i + d]` với một **độ lệch `d`** cố định. Các cặp `(i, i + d)` nằm trên một **đường chéo** của bảng `m × n`. Một mảng con chung chính là một **đoạn liên tiếp các vị trí khớp** trên một đường chéo nào đó. Duyệt hết các độ lệch `d` và tìm đoạn khớp dài nhất là một cách giải khác (cách 5).

In [7]:
a, b = [1, 2, 3, 2, 1], [3, 2, 1, 4, 7]
d = -2                                              # a[i] đối diện b[i + d], tức a[2] ~ b[0]
pairs = [(i, i + d) for i in range(len(a)) if 0 <= i + d < len(b)]
print([(i, j, a[i] == b[j]) for i, j in pairs])      # a[2..4] = [3,2,1] khớp b[0..2]

[(2, 0, True), (3, 1, True), (4, 2, True)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét `nums1 = [1,2,3,2,1]`, `nums2 = [3,2,1,4,7]`. Ta lập bảng: hàng = phần tử `nums1`, cột = phần tử `nums2`. Ô `(i, j)` ghi **độ dài đoạn chung dài nhất kết thúc tại hai vị trí này**:

- Nếu `nums1[i] != nums2[j]`: đoạn bị đứt → `0`.
- Nếu bằng nhau: đoạn dài thêm một so với đoạn kết thúc ở **ô chéo trên-trái** → `chéo + 1`.

|   | 3 | 2 | 1 | 4 | 7 |
|---|---|---|---|---|---|
| **1** | 0 | 0 | 1 | 0 | 0 |
| **2** | 0 | 1 | 0 | 0 | 0 |
| **3** | 1 | 0 | 0 | 0 | 0 |
| **2** | 0 | 2 | 0 | 0 | 0 |
| **1** | 0 | 0 | 3 | 0 | 0 |

Ô `(1 ở hàng 5, 1 ở cột 3)` bằng `3`: chuỗi các ô chéo `3 → 2 → 1` (hàng 3, 4, 5 và cột 1, 2, 3) tăng dần `1, 2, 3`. Đáp án là **số lớn nhất trong bảng** = `3`.

Quan trọng: các số khớp xếp thành **đoạn liên tiếp trên đường chéo**, và đứt ở đâu thì **về `0` ở đó**. Đó là điều phân biệt với dãy con (LCS), nơi đứt chỉ làm đoạn "không tăng" chứ không về `0`. Cell dưới tính lại bảng bằng code ngắn nhất.

In [8]:
a, b = [1, 2, 3, 2, 1], [3, 2, 1, 4, 7]
m, n = len(a), len(b)
f = [[0] * (n + 1) for _ in range(m + 1)]
for i in range(1, m + 1):
    for j in range(1, n + 1):
        if a[i - 1] == b[j - 1]:
            f[i][j] = f[i - 1][j - 1] + 1
print("      ", b)
for i in range(1, m + 1):
    print(f"{a[i - 1]:>3}:", f[i][1:])
print("đáp án = max của bảng =", max(max(row) for row in f))

       [3, 2, 1, 4, 7]
  1: [0, 0, 1, 0, 0]
  2: [0, 1, 0, 0, 0]
  3: [1, 0, 0, 0, 0]
  2: [0, 2, 0, 0, 0]
  1: [0, 0, 3, 0, 0]
đáp án = max của bảng = 3


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp vị trí bắt đầu rồi mở rộng

**Ý tưởng.** Với mỗi cặp `(i, j)` là vị trí bắt đầu trong `nums1` và `nums2`, ta mở rộng `k = 0, 1, 2, ...` chừng nào `nums1[i + k] == nums2[j + k]` (còn trong biên). Độ dài đoạn khớp tại `(i, j)` là `k`. Đáp án là `k` lớn nhất qua mọi cặp.

In [9]:
def length_brute(nums1, nums2):
    m, n = len(nums1), len(nums2)
    best = 0
    for i in range(m):
        for j in range(n):
            k = 0
            while i + k < m and j + k < n and nums1[i + k] == nums2[j + k]:
                k += 1
            best = max(best, k)
    return best


for a, b, expected in examples:
    got = length_brute(a, b)
    print(a, b, "->", got)
    assert got == expected

[1, 2, 3, 2, 1] [3, 2, 1, 4, 7] -> 3
[0, 0, 0, 0, 0] [0, 0, 0, 0, 0] -> 5
[1, 2, 3] [1, 3] -> 1
[1] [2] -> 0


**Độ phức tạp.** Có `m × n` cặp bắt đầu, mỗi cặp mở rộng tối đa `min(m, n)` bước: thời gian **O(m · n · min(m, n))** (tối đa `10^9` với `1000`), bộ nhớ **O(1)**. Với hai mảng toàn số `0` dài `1000` đây là trường hợp tệ nhất → **TLE**.

**Nút thắt.** Khi mở rộng từ `(i, j)` ta đã so `(i+1, j+1)`, `(i+2, j+2)`, ... rồi; sau đó bắt đầu lại ở `(i+1, j+1)` ta **so lại y hệt**. Ta nên nhớ "đoạn khớp kết thúc ở đây dài bao nhiêu" để dùng lại.

## 6. Ý tưởng thuật toán / bước đột phá

**Nhìn theo điểm kết thúc, không theo điểm bắt đầu.** Gọi `f[i][j]` là độ dài đoạn chung **dài nhất kết thúc tại** `nums1[i-1]` và `nums2[j-1]` (đoạn phải **bao gồm** hai phần tử này). Khi đó:

- nếu `nums1[i-1] != nums2[j-1]`: không đoạn chung nào kết thúc ở đây → `f[i][j] = 0`;
- nếu bằng nhau: đoạn chung dài nhất kết thúc ở đây là đoạn kết thúc ở `(i-1, j-1)` **kéo dài thêm** một phần tử → `f[i][j] = f[i-1][j-1] + 1`.

Đáp án = `max f[i][j]` trên cả bảng. Mỗi ô tính trong `O(1)` nên tổng `O(m · n)`.

Hai hướng khác khai thác cùng cấu trúc:

- **Tìm kiếm nhị phân trên độ dài + băm**: "có đoạn chung độ dài `L`?" là bài toán đơn điệu; kiểm tra bằng cách băm mọi cửa sổ độ dài `L` của một mảng rồi so với mảng kia.
- **Trượt đường chéo**: chạy qua mọi độ lệch `d`, tìm đoạn khớp liên tiếp dài nhất — chính là duyệt từng đường chéo của bảng, không cần lưu bảng.

## 7. Các cách tiếp theo

Tổng cộng 5 cách: cách 1 (vét cạn, mục 5) và bốn cách dưới đây.

### 7.1. Cách 2 — Quy hoạch động 2 chiều (hậu tố chung dài nhất)

**Giải thích từng đoạn.**

- Tạo `f` kích thước `(m + 1) × (n + 1)`, toàn `0`. Hàng/cột `0` là "chưa lấy phần tử nào" (đoạn rỗng, độ dài `0`) nên không cần xử lý biên riêng.
- Duyệt `i` từ `1` đến `m`, `j` từ `1` đến `n`: nếu `nums1[i-1] == nums2[j-1]` thì `f[i][j] = f[i-1][j-1] + 1`, ngược lại để `0`.
- Cập nhật đáp án `best = max(best, f[i][j])` ngay trong vòng lặp.

In [10]:
def length_dp2d(nums1, nums2):
    m, n = len(nums1), len(nums2)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    best = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if nums1[i - 1] == nums2[j - 1]:
                f[i][j] = f[i - 1][j - 1] + 1
                best = max(best, f[i][j])
    return best


for a, b, expected in examples:
    got = length_dp2d(a, b)
    print(a, b, "->", got)
    assert got == expected

[1, 2, 3, 2, 1] [3, 2, 1, 4, 7] -> 3
[0, 0, 0, 0, 0] [0, 0, 0, 0, 0] -> 5
[1, 2, 3] [1, 3] -> 1
[1] [2] -> 0


**Code nộp LeetCode (cách 2):**

In [11]:
class Solution2:
    def findLength(self, nums1: List[int], nums2: List[int]) -> int:
        m, n = len(nums1), len(nums2)
        f = [[0] * (n + 1) for _ in range(m + 1)]
        ans = 0
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if nums1[i - 1] == nums2[j - 1]:
                    f[i][j] = f[i - 1][j - 1] + 1
                    ans = max(ans, f[i][j])
        return ans


for a, b, expected in examples:
    assert Solution2().findLength(a, b) == expected
print("Solution2 OK")

Solution2 OK


**Độ phức tạp cách 2.** Thời gian **O(m · n)** (`10^6` ô với `1000 × 1000`), bộ nhớ **O(m · n)** cho bảng.

### 7.2. Cách 3 — Rút gọn còn **một hàng** (O(n) bộ nhớ)

**Ý tưởng.** `f[i][j]` chỉ dùng hàng `i - 1` (ô `f[i-1][j-1]`). Ta giữ **một** mảng `f` độ dài `n + 1` và cập nhật tại chỗ. Vì ô `j` cần giá trị **cũ** của ô `j - 1` (thuộc hàng trước), ta phải duyệt `j` **từ phải sang trái** (`n` xuống `1`): khi tính `f[j]`, ô `f[j - 1]` còn là giá trị của hàng trước. Và khi hai phần tử **không bằng nhau** phải **gán `f[j] = 0`** (nếu không giá trị cũ của hàng trước còn sót lại).

In [12]:
def length_dp1d(nums1, nums2, verbose=False):
    n = len(nums2)
    f = [0] * (n + 1)
    best = 0
    for i, x in enumerate(nums1, 1):
        for j in range(n, 0, -1):               # phải -> trái: f[j-1] vẫn là của hàng trước
            if x == nums2[j - 1]:
                f[j] = f[j - 1] + 1
                best = max(best, f[j])
            else:
                f[j] = 0                         # đứt -> về 0 (bắt buộc)
        if verbose:
            print(f"  sau hàng {i} (x = {x}): f = {f[1:]}")
    return best


for a, b, expected in examples:
    got = length_dp1d(a, b, verbose=(a == examples[0][0]))
    assert got == expected
print("OK")

  sau hàng 1 (x = 1): f = [0, 0, 1, 0, 0]
  sau hàng 2 (x = 2): f = [0, 1, 0, 0, 0]
  sau hàng 3 (x = 3): f = [1, 0, 0, 0, 0]
  sau hàng 4 (x = 2): f = [0, 2, 0, 0, 0]
  sau hàng 5 (x = 1): f = [0, 0, 3, 0, 0]
OK


**Code nộp LeetCode (cách 3):**

In [13]:
class Solution3:
    def findLength(self, nums1: List[int], nums2: List[int]) -> int:
        n = len(nums2)
        f = [0] * (n + 1)
        ans = 0
        for x in nums1:
            for j in range(n, 0, -1):
                if x == nums2[j - 1]:
                    f[j] = f[j - 1] + 1
                    ans = max(ans, f[j])
                else:
                    f[j] = 0
        return ans


for a, b, expected in examples:
    assert Solution3().findLength(a, b) == expected
print("Solution3 OK")

Solution3 OK


**Độ phức tạp cách 3.** Thời gian **O(m · n)**, bộ nhớ **O(n)**. (Có thể chọn mảng ngắn hơn làm `nums2` để bộ nhớ là `O(min(m, n))`.)

### 7.3. Cách 4 — Tìm kiếm nhị phân độ dài + băm cuộn

**Ý tưởng.** Gọi `P(L)` = "tồn tại mảng con độ dài `L` chung". `P(0)` luôn đúng, và `P` đơn điệu (đúng với `L` thì đúng với `L - 1`). Ta chặt nhị phân tìm `L` lớn nhất với `P(L)` đúng, `L ∈ [0, min(m, n)]`.

**Kiểm tra `P(L)`** (hàm `check(L)`): tính băm mọi cửa sổ độ dài `L` của `nums1`, cất vào dict `băm → danh sách vị trí bắt đầu`. Với mỗi cửa sổ của `nums2`, nếu băm có trong dict thì **so lại thật** từng ứng viên (`nums1[i:i+L] == nums2[j:j+L]`) để loại va chạm. Có một cặp khớp thật thì `P(L)` đúng.

In [14]:
def length_binary_hash(nums1, nums2, verbose=False):
    m, n = len(nums1), len(nums2)
    P1, P2 = prefix_hashes(nums1), prefix_hashes(nums2)
    pw = [1]
    for _ in range(min(m, n)):
        pw.append(pw[-1] * BASE % MOD)

    def check(L):
        if L == 0:
            return True
        seen = {}
        for i in range(m - L + 1):
            seen.setdefault(window_hash(P1, pw, i, L), []).append(i)
        for j in range(n - L + 1):
            for i in seen.get(window_hash(P2, pw, j, L), ()):
                if nums1[i:i + L] == nums2[j:j + L]:         # kiểm chứng: loại va chạm
                    return True
        return False

    lo, hi = 0, min(m, n)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        ok = check(mid)
        if verbose:
            print(f"  lo = {lo}, hi = {hi}, mid = {mid} -> có đoạn chung độ dài {mid}? {ok}")
        if ok:
            lo = mid
        else:
            hi = mid - 1
    return lo


for a, b, expected in examples:
    got = length_binary_hash(a, b, verbose=(a == examples[0][0]))
    assert got == expected
print("OK")

  lo = 0, hi = 5, mid = 3 -> có đoạn chung độ dài 3? True
  lo = 3, hi = 5, mid = 4 -> có đoạn chung độ dài 4? False
OK


**Code nộp LeetCode (cách 4):**

In [15]:
class Solution4:
    def findLength(self, nums1: List[int], nums2: List[int]) -> int:
        mod, base = (1 << 61) - 1, 131
        m, n = len(nums1), len(nums2)

        def prefix(a):
            h = [0]
            for x in a:
                h.append((h[-1] * base + x + 1) % mod)
            return h

        h1, h2 = prefix(nums1), prefix(nums2)
        pw = [1]
        for _ in range(min(m, n)):
            pw.append(pw[-1] * base % mod)

        def check(L: int) -> bool:
            if L == 0:
                return True
            seen = {}
            for i in range(m - L + 1):
                key = (h1[i + L] - h1[i] * pw[L]) % mod
                seen.setdefault(key, []).append(i)
            for j in range(n - L + 1):
                key = (h2[j + L] - h2[j] * pw[L]) % mod
                for i in seen.get(key, ()):
                    if nums1[i:i + L] == nums2[j:j + L]:
                        return True
            return False

        lo, hi = 0, min(m, n)
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if check(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo


for a, b, expected in examples:
    assert Solution4().findLength(a, b) == expected
print("Solution4 OK")

Solution4 OK


**Độ phức tạp cách 4.** Mỗi `check(L)` tốn `O(m + n)` băm (cộng thời gian so lại các ứng viên; thường ít), có `O(log min(m, n))` lần kiểm tra: thời gian **O((m + n) log min(m, n))** kỳ vọng; bộ nhớ **O(m + n)**. Nhanh nhất về mặt tiệm cận, nhưng phức tạp để cài và phải xử lý va chạm băm.

### 7.4. Cách 5 — Trượt hai mảng lệch nhau, đếm đoạn khớp dài nhất trên mỗi đường chéo

**Ý tưởng.** Cố định độ lệch `d = j - i`, so `nums1[i]` với `nums2[i + d]` cho mọi `i` hợp lệ, và đếm **đoạn khớp liên tiếp dài nhất** (biến `run` tăng khi khớp, về `0` khi lệch). `d` chạy từ `-(m - 1)` đến `n - 1`. Không cần bảng; bộ nhớ `O(1)`.

In [16]:
def length_diagonal(nums1, nums2, verbose=False):
    m, n = len(nums1), len(nums2)
    best = 0
    for d in range(-(m - 1), n):                 # nums1[i] đối diện nums2[i + d]
        run = 0
        for i in range(max(0, -d), min(m, n - d)):
            if nums1[i] == nums2[i + d]:
                run += 1
                best = max(best, run)
            else:
                run = 0
        if verbose:
            print(f"  d = {d:>2}: best sau khi quét đường chéo này = {best}")
    return best


for a, b, expected in examples:
    got = length_diagonal(a, b, verbose=(a == examples[0][0]))
    assert got == expected
print("OK")

  d = -4: best sau khi quét đường chéo này = 0
  d = -3: best sau khi quét đường chéo này = 0
  d = -2: best sau khi quét đường chéo này = 3
  d = -1: best sau khi quét đường chéo này = 3
  d =  0: best sau khi quét đường chéo này = 3
  d =  1: best sau khi quét đường chéo này = 3
  d =  2: best sau khi quét đường chéo này = 3
  d =  3: best sau khi quét đường chéo này = 3
  d =  4: best sau khi quét đường chéo này = 3
OK


**Code nộp LeetCode (cách 5):**

In [17]:
class Solution5:
    def findLength(self, nums1: List[int], nums2: List[int]) -> int:
        m, n = len(nums1), len(nums2)
        ans = 0
        for d in range(-(m - 1), n):
            run = 0
            for i in range(max(0, -d), min(m, n - d)):
                if nums1[i] == nums2[i + d]:
                    run += 1
                    if run > ans:
                        ans = run
                else:
                    run = 0
        return ans


for a, b, expected in examples:
    assert Solution5().findLength(a, b) == expected
print("Solution5 OK")

Solution5 OK


**Độ phức tạp cách 5.** Có `m + n - 1` đường chéo, tổng số ô duyệt là `m · n`: thời gian **O(m · n)**; bộ nhớ **O(1)**. Cách này chính là cách 2 nhưng duyệt theo đường chéo và chỉ giữ `run` thay vì cả bảng.

## 8. TRACE từng bước

### 8.1. Trace cách 2 (bảng DP) trên ví dụ 1

Bảng trạng thái: mỗi dòng in ô `(i, j)` **khi hai phần tử bằng nhau**, kèm ô chéo `f[i-1][j-1]`:

| Cột | Ý nghĩa |
|---|---|
| `(i, j)` | chỉ số trong bảng (1-based) |
| `nums1[i-1]`, `nums2[j-1]` | hai phần tử được so |
| `f[i-1][j-1]` | giá trị ô chéo trên-trái |
| `f[i][j]` | giá trị mới (`chéo + 1`) |

In [18]:
def trace_dp2d(nums1, nums2):
    m, n = len(nums1), len(nums2)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    best = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if nums1[i - 1] == nums2[j - 1]:
                f[i][j] = f[i - 1][j - 1] + 1
                best = max(best, f[i][j])
                print(f"({i},{j}): nums1[{i - 1}] = nums2[{j - 1}] = {nums1[i - 1]} | chéo f[{i - 1}][{j - 1}] = {f[i - 1][j - 1]} -> f[{i}][{j}] = {f[i][j]} | best = {best}")
    print("bảng cuối cùng (bỏ hàng/cột 0):")
    print("      ", nums2)
    for i in range(1, m + 1):
        print(f"{nums1[i - 1]:>3}:", f[i][1:])
    return best

print("kết quả:", trace_dp2d([1, 2, 3, 2, 1], [3, 2, 1, 4, 7]))

(1,3): nums1[0] = nums2[2] = 1 | chéo f[0][2] = 0 -> f[1][3] = 1 | best = 1
(2,2): nums1[1] = nums2[1] = 2 | chéo f[1][1] = 0 -> f[2][2] = 1 | best = 1
(3,1): nums1[2] = nums2[0] = 3 | chéo f[2][0] = 0 -> f[3][1] = 1 | best = 1
(4,2): nums1[3] = nums2[1] = 2 | chéo f[3][1] = 1 -> f[4][2] = 2 | best = 2
(5,3): nums1[4] = nums2[2] = 1 | chéo f[4][2] = 2 -> f[5][3] = 3 | best = 3
bảng cuối cùng (bỏ hàng/cột 0):
       [3, 2, 1, 4, 7]
  1: [0, 0, 1, 0, 0]
  2: [0, 1, 0, 0, 0]
  3: [1, 0, 0, 0, 0]
  2: [0, 2, 0, 0, 0]
  1: [0, 0, 3, 0, 0]
kết quả: 3


Đọc trace: `f[3][1] = 1` (cặp `3` với `3`); ô `(4, 2)` (`2` với `2`) có chéo `f[3][1] = 1` nên được `2`; ô `(5, 3)` (`1` với `1`) có chéo `f[4][2] = 2` nên được `3`: ba ô xếp thành một đường chéo `3 → 2 → 1`. Hai ô khớp "cô lập" `(1,3)` và `(2,2)` chỉ được `1` vì chéo của chúng bằng `0`.

### 8.2. Trace cách 3 (một hàng, duyệt `j` từ phải sang trái)

In `f` sau mỗi phần tử của `nums1`, và đánh dấu từng bước khớp trong hàng đó:

In [19]:
def trace_dp1d(nums1, nums2):
    n = len(nums2)
    f = [0] * (n + 1)
    best = 0
    print("nums2:", nums2)
    for i, x in enumerate(nums1, 1):
        steps = []
        for j in range(n, 0, -1):
            if x == nums2[j - 1]:
                f[j] = f[j - 1] + 1
                best = max(best, f[j])
                steps.append(f"j={j}: f[{j}] = f[{j - 1}] + 1 = {f[j]}")
            else:
                f[j] = 0
        print(f"hàng {i} (x = {x}): f = {f[1:]} | {'; '.join(steps) if steps else 'không khớp'}")
    return best

print("kết quả:", trace_dp1d([1, 2, 3, 2, 1], [3, 2, 1, 4, 7]))

nums2: [3, 2, 1, 4, 7]
hàng 1 (x = 1): f = [0, 0, 1, 0, 0] | j=3: f[3] = f[2] + 1 = 1
hàng 2 (x = 2): f = [0, 1, 0, 0, 0] | j=2: f[2] = f[1] + 1 = 1
hàng 3 (x = 3): f = [1, 0, 0, 0, 0] | j=1: f[1] = f[0] + 1 = 1
hàng 4 (x = 2): f = [0, 2, 0, 0, 0] | j=2: f[2] = f[1] + 1 = 2
hàng 5 (x = 1): f = [0, 0, 3, 0, 0] | j=3: f[3] = f[2] + 1 = 3
kết quả: 3


Quan sát: ở hàng 4 (`x = 2`), duyệt `j = 2` dùng `f[1]` **cũ** (giá trị `1` của hàng 3) nên được `2`. Nếu duyệt `j` từ trái sang phải, `f[1]` đã bị ghi đè trước đó → sai (mục 13).

### 8.3. Trace cách 4 (nhị phân độ dài + băm)

Ví dụ 1 có `min(m, n) = 5`; ta in từng bước thu hẹp `[lo, hi]` và mảng con chung tìm được ở mỗi `mid`:

In [20]:
def trace_binary(nums1, nums2):
    m, n = len(nums1), len(nums2)

    def common_of_len(L):
        """Trả về một mảng con chung độ dài L (hoặc None), để thấy bằng chứng."""
        seen = {}
        for i in range(m - L + 1):
            seen.setdefault(tuple(nums1[i:i + L]), i)
        for j in range(n - L + 1):
            t = tuple(nums2[j:j + L])
            if t in seen:
                return t, seen[t], j
        return None

    lo, hi = 0, min(m, n)
    step = 0
    while lo < hi:
        mid = (lo + hi + 1) // 2
        found = common_of_len(mid)
        step += 1
        if found:
            print(f"bước {step}: [lo={lo}, hi={hi}] mid={mid} -> CÓ {found[0]} (nums1 từ {found[1]}, nums2 từ {found[2]}) -> lo = {mid}")
            lo = mid
        else:
            print(f"bước {step}: [lo={lo}, hi={hi}] mid={mid} -> KHÔNG có -> hi = {mid - 1}")
            hi = mid - 1
    return lo

print("kết quả:", trace_binary([1, 2, 3, 2, 1], [3, 2, 1, 4, 7]))

bước 1: [lo=0, hi=5] mid=3 -> CÓ (3, 2, 1) (nums1 từ 2, nums2 từ 0) -> lo = 3
bước 2: [lo=3, hi=5] mid=4 -> KHÔNG có -> hi = 3
kết quả: 3


Đọc trace: `mid = 3` → có `[3,2,1]` nên `lo = 3`; `mid = 4` → không có → `hi = 3`; `lo == hi = 3` dừng. Chỉ cần `2` lần kiểm tra thay vì thử mọi `L`.

### 8.4. Trace cách 5 (đường chéo)

Phần in `verbose=True` ở mục 7.4 cho thấy `best` thay đổi theo từng độ lệch `d`. Ở đây chỉ in các đường chéo có khớp, kèm đoạn khớp dài nhất trên đường chéo đó:

In [21]:
def trace_diagonal(nums1, nums2):
    m, n = len(nums1), len(nums2)
    best = 0
    for d in range(-(m - 1), n):
        run, local, end_i = 0, 0, -1
        for i in range(max(0, -d), min(m, n - d)):
            if nums1[i] == nums2[i + d]:
                run += 1
                if run > local:
                    local, end_i = run, i
            else:
                run = 0
        if local:
            seg = nums1[end_i - local + 1:end_i + 1]
            print(f"d = {d:>2}: đoạn khớp dài nhất = {local}  {seg}  (nums1[{end_i - local + 1}..{end_i}] ~ nums2[{end_i - local + 1 + d}..{end_i + d}])")
        best = max(best, local)
    return best

print("kết quả:", trace_diagonal([1, 2, 3, 2, 1], [3, 2, 1, 4, 7]))

d = -2: đoạn khớp dài nhất = 3  [3, 2, 1]  (nums1[2..4] ~ nums2[0..2])
d =  0: đoạn khớp dài nhất = 1  [2]  (nums1[1..1] ~ nums2[1..1])
d =  2: đoạn khớp dài nhất = 1  [1]  (nums1[0..0] ~ nums2[2..2])
kết quả: 3


Đường chéo `d = -2` là đường duy nhất cho đoạn dài `3` (`[3,2,1]`); các đường chéo khác chỉ có các khớp lẻ tẻ dài `1`.

## 9. Code nộp LeetCode

Chọn **cách 3 (DP một hàng)**: code ngắn, `O(m · n)` thời gian và `O(n)` bộ nhớ, không phải lo va chạm băm. Đây là code đúng class/method/signature (trên LeetCode `List` có sẵn; ở đây ta đã import ở trên):

In [22]:
class Solution:
    def findLength(self, nums1: List[int], nums2: List[int]) -> int:
        n = len(nums2)
        f = [0] * (n + 1)            # f[j]: độ dài đoạn chung dài nhất KẾT THÚC tại nums2[j-1] (với phần tử hiện tại của nums1)
        ans = 0
        for x in nums1:
            for j in range(n, 0, -1):    # phải -> trái để f[j-1] vẫn là giá trị của hàng trước
                if x == nums2[j - 1]:
                    f[j] = f[j - 1] + 1
                    ans = max(ans, f[j])
                else:
                    f[j] = 0             # đứt đoạn -> về 0
        return ans


for a, b, expected in examples:
    got = Solution().findLength(a, b)
    print(a, b, "->", got)
    assert got == expected

[1, 2, 3, 2, 1] [3, 2, 1, 4, 7] -> 3
[0, 0, 0, 0, 0] [0, 0, 0, 0, 0] -> 5
[1, 2, 3] [1, 3] -> 1
[1] [2] -> 0


## 10. Vì sao thuật toán đúng

**Định nghĩa.** Với `1 ≤ i ≤ m`, `1 ≤ j ≤ n`, gọi `f[i][j]` là **độ dài lớn nhất `k`** sao cho `nums1[i-k:i] == nums2[j-k:j]` (hai mảng con kết thúc đúng tại `i-1` và `j-1`, độ dài `k`; `k = 0` luôn hợp lệ).

**Công thức.**

- Nếu `nums1[i-1] != nums2[j-1]`: mọi `k ≥ 1` đều sai vì phần tử cuối không khớp, nên `f[i][j] = 0`.
- Nếu `nums1[i-1] == nums2[j-1]`: với `k ≥ 1`, điều kiện `nums1[i-k:i] == nums2[j-k:j]` tương đương `nums1[i-k:i-1] == nums2[j-k:j-1]` (độ dài `k - 1`, kết thúc tại `(i-1, j-1)`). Tập các `k - 1` thỏa mãn chính là `0..f[i-1][j-1]` (vì nếu một đoạn chung dài `t` kết thúc ở đó thì mọi `t' < t` cũng vậy — cắt bớt phần đầu). Do đó `k` lớn nhất là `f[i-1][j-1] + 1`.

**Đáp án.** Một mảng con chung bất kỳ kết thúc ở một cặp `(i, j)` nào đó, nên độ dài lớn nhất toàn cục là `max f[i][j]` (hoặc `0` nếu không có phần tử chung).

**Cách 3 (một hàng).** Hàng `i` của bảng chỉ phụ thuộc vào hàng `i - 1`. Khi duyệt `j` từ `n` xuống `1`, tại thời điểm tính `f[j]` ô `f[j - 1]` chưa bị ghi đè trong vòng này nên vẫn là `f[i-1][j-1]`; còn ô `f[j]` cũ (hàng `i-1`) bị thay bằng giá trị hàng `i`. Việc gán `0` khi không khớp là cần thiết để xóa giá trị cũ.

**Cách 5 (đường chéo).** Ô `(i, j)` chỉ phụ thuộc ô chéo `(i-1, j-1)` nên cùng một đường chéo `d = j - i`; `run` chính là `f` dọc theo đường chéo.

**Cách 4 (nhị phân + băm).** Đơn điệu: nếu có mảng con chung độ dài `L` thì cắt phần tử đầu ta có mảng con chung độ dài `L - 1`. Vậy tập các `L` thỏa `P(L)` là `{0, ..., L*}` và nhị phân tìm đúng `L*`. Hàm `check(L)` so lại bằng `==` sau khi băm trùng nên không bị sai vì va chạm (băm chỉ để lọc nhanh).

**Cách 1** đúng theo định nghĩa (thử mọi điểm bắt đầu và mở rộng tối đa).

## 11. Phân tích Time / Space Complexity

Gọi `m = len(nums1)`, `n = len(nums2)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn mọi cặp bắt đầu | O(m · n · min(m, n)) | O(1) | tệ nhất với mảng đồng nhất (toàn `0`) |
| 2. DP 2 chiều | O(m · n) | O(m · n) | dễ hiểu, in được cả bảng |
| 3. DP một hàng | O(m · n) | O(n) | **bản nộp** |
| 4. Nhị phân độ dài + băm | O((m + n) log min(m, n)) | O(m + n) | nhanh nhất tiệm cận, cài phức tạp |
| 5. Trượt đường chéo | O(m · n) | O(1) | không cần bảng |

**Chi tiết.** Với `m = n = 1000`: cách 2/3/5 thực hiện `10^6` phép so sánh (khoảng nửa giây trong Python); cách 1 trên hai mảng toàn `0` cần khoảng `3 × 10^8` bước nên TLE; cách 4 chỉ băm vài nghìn cửa sổ mỗi lần kiểm tra.

**Chọn mảng ngắn làm `nums2`** trong cách 3 để bộ nhớ chỉ `O(min(m, n))`.

## 12. Bộ test

Ta kiểm tra theo nhiều lớp:

1. Ví dụ của đề (và các ví dụ tự đặt) cho cả 5 cách + bản nộp.
2. Edge case: mảng một phần tử, không phần tử chung, toàn bộ giống nhau, một mảng là mảng con của mảng kia, hai mảng lệch độ dài, hai đoạn khớp dài bằng nhau.
3. **Phản ví dụ** của các cách làm sai.
4. Đối chiếu với **oracle độc lập** (liệt kê mọi mảng con của `nums1` vào `set` các `tuple`, rồi thử từng độ dài giảm dần trên `nums2`) trên input ngẫu nhiên có seed.
5. Test lớn `1000 × 1000` và so sánh thời gian.

In [23]:
def oracle_sets(nums1, nums2):
    """Oracle độc lập: tập mọi mảng con của nums1 (dưới dạng tuple); tìm độ dài lớn nhất xuất hiện trong nums2."""
    all1 = {tuple(nums1[i:j]) for i in range(len(nums1)) for j in range(i + 1, len(nums1) + 1)}
    best = 0
    for i in range(len(nums2)):
        for j in range(i + 1, len(nums2) + 1):
            if j - i > best and tuple(nums2[i:j]) in all1:
                best = j - i
    return best


for a, b, expected in examples:
    assert oracle_sets(a, b) == expected
print("oracle khớp ví dụ")

solvers = {
    "brute": length_brute,
    "dp2d": length_dp2d,
    "dp1d": length_dp1d,
    "binary_hash": length_binary_hash,
    "diagonal": length_diagonal,
    "Solution2": lambda a, b: Solution2().findLength(a, b),
    "Solution3": lambda a, b: Solution3().findLength(a, b),
    "Solution4": lambda a, b: Solution4().findLength(a, b),
    "Solution5": lambda a, b: Solution5().findLength(a, b),
    "Solution": lambda a, b: Solution().findLength(a, b),
}
for a, b, expected in examples:
    for name, fn in solvers.items():
        assert fn(list(a), list(b)) == expected, (name, a, b)
print("tất cả cách khớp ví dụ:", list(solvers))

oracle khớp ví dụ
tất cả cách khớp ví dụ: ['brute', 'dp2d', 'dp1d', 'binary_hash', 'diagonal', 'Solution2', 'Solution3', 'Solution4', 'Solution5', 'Solution']


In [24]:
# --- Edge case ---
edge_cases = [
    ([5], [5]),                                      # một phần tử, giống
    ([5], [6]),                                      # một phần tử, khác
    ([1, 2, 3], [4, 5, 6]),                          # không phần tử chung
    ([1, 2, 3], [1, 2, 3]),                          # giống hệt
    ([1, 2, 3, 4], [2, 3]),                          # mảng ngắn là mảng con của mảng dài
    ([2, 3], [1, 2, 3, 4]),                          # đảo vai trò
    ([1, 2, 3], [1, 3]),                             # dãy con chung dài 2 nhưng mảng con chung chỉ 1
    ([1, 2, 1, 2, 1], [2, 1, 2]),                    # lặp lại
    ([0, 0, 0], [0, 0, 0, 0, 0]),                    # toàn 0, độ dài khác nhau
    ([1, 2, 3, 9, 1, 2, 3, 4], [1, 2, 3, 4, 5]),     # có hai đoạn gần giống: dài hơn là [1,2,3,4]
    ([1, 2, 3, 4], [4, 3, 2, 1]),                    # đảo ngược -> 1
    ([100, 0, 100], [0, 100, 0]),                    # biên giá trị
    ([1, 2], [2, 1]),                                # chỉ khớp 1
]
for a, b in edge_cases:
    want = oracle_sets(a, b)
    for name, fn in solvers.items():
        got = fn(list(a), list(b))
        assert got == want, (name, a, b, got, want)
    print(a, b, "->", want)

[5] [5] -> 1
[5] [6] -> 0
[1, 2, 3] [4, 5, 6] -> 0
[1, 2, 3] [1, 2, 3] -> 3
[1, 2, 3, 4] [2, 3] -> 2
[2, 3] [1, 2, 3, 4] -> 2
[1, 2, 3] [1, 3] -> 1
[1, 2, 1, 2, 1] [2, 1, 2] -> 3
[0, 0, 0] [0, 0, 0, 0, 0] -> 3
[1, 2, 3, 9, 1, 2, 3, 4] [1, 2, 3, 4, 5] -> 4
[1, 2, 3, 4] [4, 3, 2, 1] -> 1
[100, 0, 100] [0, 100, 0] -> 2
[1, 2] [2, 1] -> 1


**Phản ví dụ.** Các cách "trông có vẻ đúng" nhưng sai:

- *Sai A:* dùng công thức **LCS** (dãy con): `f[i][j] = max(f[i-1][j], f[i][j-1])` khi không khớp.
- *Sai B:* DP một hàng nhưng duyệt `j` **tăng dần**.
- *Sai C:* DP một hàng nhưng **không gán `0`** khi lệch.
- *Sai D:* trả `f[m][n]` (ô cuối) thay vì `max` cả bảng.

In [25]:
def wrong_lcs(a, b):                               # sai A: dãy con, không phải mảng con
    m, n = len(a), len(b)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            f[i][j] = f[i - 1][j - 1] + 1 if a[i - 1] == b[j - 1] else max(f[i - 1][j], f[i][j - 1])
    return f[m][n]

def wrong_ascending(a, b):                         # sai B
    n = len(b)
    f = [0] * (n + 1)
    best = 0
    for x in a:
        for j in range(1, n + 1):                  # trái -> phải: f[j-1] đã bị ghi đè trong vòng này
            if x == b[j - 1]:
                f[j] = f[j - 1] + 1
                best = max(best, f[j])
            else:
                f[j] = 0
    return best

def wrong_no_reset(a, b):                          # sai C
    n = len(b)
    f = [0] * (n + 1)
    best = 0
    for x in a:
        for j in range(n, 0, -1):
            if x == b[j - 1]:
                f[j] = f[j - 1] + 1
                best = max(best, f[j])
            # THIẾU: else: f[j] = 0
    return best

def wrong_last_cell(a, b):                         # sai D
    m, n = len(a), len(b)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                f[i][j] = f[i - 1][j - 1] + 1
    return f[m][n]


tests = {
    "A": (wrong_lcs, [1, 2, 3], [1, 3]),                  # đúng 1, sai ra 2
    "B": (wrong_ascending, [1], [1, 1, 1]),               # đúng 1, sai ra 3
    "D": (wrong_last_cell, [1, 2, 3], [1, 2, 4]),         # đúng 2, sai ra 0
}
for label, (fn, a, b) in tests.items():
    print(f"sai {label}: {fn(a, b)} | đúng: {oracle_sets(a, b)} | a = {a}, b = {b}")
    assert fn(a, b) != oracle_sets(a, b)

# Sai C: tìm một phản ví dụ nhỏ bằng tìm kiếm có seed
rng = random.Random(718)
foundC = None
for _ in range(20000):
    a = [rng.randint(0, 2) for _ in range(rng.randint(2, 5))]
    b = [rng.randint(0, 2) for _ in range(rng.randint(2, 5))]
    if wrong_no_reset(a, b) != oracle_sets(a, b):
        foundC = (a, b)
        break
print("sai C:", foundC, "-> sai ra", wrong_no_reset(*foundC), "| đúng", oracle_sets(*foundC))
assert foundC is not None

sai A: 2 | đúng: 1 | a = [1, 2, 3], b = [1, 3]
sai B: 3 | đúng: 1 | a = [1], b = [1, 1, 1]
sai D: 0 | đúng: 2 | a = [1, 2, 3], b = [1, 2, 4]
sai C: ([2, 0, 1, 1], [1, 2, 1]) -> sai ra 2 | đúng 1


Giải thích: **A** cho `2` vì coi dãy con `[1,3]` là chung, trong khi trong `nums1` hai phần tử `1` và `3` không kề nhau. **B** duyệt `j` tăng dần nên `f[j - 1]` đã bị cập nhật trong **cùng một vòng** → chuỗi khớp `[1]` bị cộng dồn thành `3` (như thể có `[1,1,1]` ở `nums1`). **C** không xóa giá trị cũ của hàng trước, nên "đoạn" bị đứt vẫn được nối tiếp bằng dữ liệu sót lại. **D** chỉ nhìn ô cuối, bỏ lỡ đoạn chung nằm ở giữa.

**Đối chiếu nhiều input ngẫu nhiên (có seed, đúng ràng buộc).** Ta sinh hai mảng với bảng chữ nhỏ (hay khớp, hay có đoạn dài), bảng chữ lớn (ít khớp), và có cả trường hợp **cấy sẵn một đoạn chung** để đáp án thường lớn hơn `1`.

In [26]:
def random_pair(rng, alpha, n1, n2, plant):
    a = [rng.randint(0, alpha) for _ in range(n1)]
    b = [rng.randint(0, alpha) for _ in range(n2)]
    if plant and n1 and n2:                                   # cấy một đoạn chung độ dài ngẫu nhiên
        L = rng.randint(1, min(n1, n2))
        seg = [rng.randint(0, alpha) for _ in range(L)]
        i, j = rng.randint(0, n1 - L), rng.randint(0, n2 - L)
        a[i:i + L] = seg
        b[j:j + L] = seg
    return a, b


rng = random.Random(7180)
count = 0
for _ in range(600):
    alpha = rng.choice([1, 2, 5, 100])
    a, b = random_pair(rng, alpha, rng.randint(1, 12), rng.randint(1, 12), rng.random() < 0.5)
    want = oracle_sets(a, b)
    for name, fn in solvers.items():
        a0, b0 = list(a), list(b)
        got = fn(a0, b0)
        assert got == want, (name, a, b, got, want)
        assert a0 == a and b0 == b, (name, "làm thay đổi input")
    count += 1
print("đối chiếu", count, "input ngẫu nhiên (n <= 12): tất cả cách khớp oracle tập tuple và không sửa input")

# n trung bình: bỏ brute (chậm), oracle vẫn chạy được
for _ in range(80):
    alpha = rng.choice([1, 2, 3, 100])
    a, b = random_pair(rng, alpha, rng.randint(13, 60), rng.randint(13, 60), rng.random() < 0.7)
    want = oracle_sets(a, b)
    for name, fn in solvers.items():
        if name == "brute":
            continue
        assert fn(list(a), list(b)) == want, (name, len(a), len(b))
print("80 input n từ 13 đến 60: OK")

đối chiếu

 600 input ngẫu nhiên (n <= 12): tất cả cách khớp oracle tập tuple và không sửa input


80 input n từ 13 đến 60: OK

In [27]:
# Test lớn nhất theo ràng buộc: 1000 x 1000 (bỏ brute, oracle_sets quá chậm; đối chiếu chéo giữa các cách)
rng = random.Random(1000)
for alpha in (1, 3, 100):
    a, b = random_pair(rng, alpha, 1000, 1000, True)
    results = {}
    for name in ("dp2d", "dp1d", "binary_hash", "diagonal", "Solution3", "Solution4", "Solution5", "Solution"):
        t0 = time.perf_counter()
        results[name] = solvers[name](list(a), list(b))
        dt = (time.perf_counter() - t0) * 1000
        print(f"alpha = {alpha:>3} | {name:>11}: {dt:8.1f} ms -> {results[name]}")
    assert len(set(results.values())) == 1, results

# trường hợp tệ nhất cho brute: toàn 0, độ dài 1000 -> đáp án 1000
zeros = [0] * 1000
for name in ("dp2d", "dp1d", "binary_hash", "diagonal", "Solution"):
    assert solvers[name](list(zeros), list(zeros)) == 1000, name
print("1000 số 0 vs 1000 số 0 -> 1000: OK")

# brute chỉ thử trên cỡ nhỏ hơn vì O(m*n*min)
z = [0] * 150
t0 = time.perf_counter()
r = length_brute(z, z)
print(f"brute trên 150 số 0: {(time.perf_counter() - t0) * 1000:.1f} ms -> {r}")
assert r == 150

alpha =   1 |        dp2d:    176.4 ms -> 305


alpha =   1 |        dp1d:     76.6 ms -> 305
alpha =   1 | binary_hash:      5.2 ms -> 305


alpha =   1 |    diagonal:    135.5 ms -> 305


alpha =   1 |   Solution3:    139.4 ms -> 305
alpha =   1 |   Solution4:      4.7 ms -> 305


alpha =   1 |   Solution5:     88.0 ms -> 305
alpha =   1 |    Solution:     78.3 ms -> 305


alpha =   3 |        dp2d:    140.4 ms -> 892
alpha =   3 |        dp1d:     60.9 ms -> 892
alpha =   3 | binary_hash:      1.6 ms -> 892
alpha =   3 |    diagonal:     54.3 ms -> 892
alpha =   3 |   Solution3:     61.4 ms -> 892
alpha =   3 |   Solution4:      1.4 ms -> 892


alpha =   3 |   Solution5:     52.1 ms -> 892
alpha =   3 |    Solution:     61.9 ms -> 892
alpha = 100 |        dp2d:     56.1 ms -> 444
alpha = 100 |        dp1d:     41.6 ms -> 444
alpha = 100 | binary_hash:      4.5 ms -> 444


alpha = 100 |    diagonal:     42.2 ms -> 444
alpha = 100 |   Solution3:     41.2 ms -> 444
alpha = 100 |   Solution4:      4.2 ms -> 444
alpha = 100 |   Solution5:     41.2 ms -> 444
alpha = 100 |    Solution:     41.2 ms -> 444


1000 số 0 vs 1000 số 0 -> 1000: OK
brute trên 150 số 0: 98.7 ms -> 150


## 13. Các lỗi người mới hay gặp

### Lỗi 1: nhầm với bài **dãy con chung dài nhất (LCS)**

LCS cho phép **bỏ qua** phần tử; bài này đòi **liên tiếp**. Công thức LCS khi không khớp là `max(f[i-1][j], f[i][j-1])` (nhớ kết quả cũ), còn bài này khi không khớp phải **về `0`**.

In [28]:
a, b = [1, 2, 3], [1, 3]
print("sai  (LCS):", wrong_lcs(a, b))
print("đúng      :", Solution().findLength(a, b))

sai  (LCS): 2
đúng      : 1


**Vì sao sai.** `[1, 3]` là dãy con chung nhưng trong `nums1` hai phần tử không kề nhau nên **không** phải mảng con. **Cách sửa:** khi `nums1[i-1] != nums2[j-1]`, đặt `f[i][j] = 0` (đoạn bị đứt).

### Lỗi 2: rút gọn về một hàng nhưng duyệt `j` **tăng dần**

Ô `f[j]` cần giá trị **cũ** của `f[j - 1]` (hàng trước). Duyệt trái sang phải thì `f[j - 1]` đã bị ghi đè bằng giá trị hàng hiện tại.

In [29]:
a, b = [1], [1, 1, 1]
print("sai  (trái->phải):", wrong_ascending(a, b))      # 3: coi như có [1,1,1] trong nums1
print("đúng             :", Solution().findLength(a, b))

sai  (trái->phải): 3
đúng             : 1


**Vì sao sai.** `nums1 = [1]` chỉ có một phần tử nên đáp án tối đa là `1`. Duyệt tăng dần: `f[1] = 1`, rồi `f[2] = f[1] + 1 = 2` (dùng `f[1]` **mới**), `f[3] = 3`. **Cách sửa:** duyệt `j` từ `n` xuống `1` (hoặc dùng hai mảng `prev`/`cur`, hoặc giữ biến tạm `diag`).

### Lỗi 3: quên gán `0` khi lệch (trong bản một hàng)

Với bảng 2 chiều mới tạo toàn `0` thì ô lệch tự nhiên là `0`; nhưng ở bản một hàng ô `f[j]` **đang chứa giá trị của hàng trước**, nếu không xóa nó thì "đoạn đã đứt" vẫn sống tiếp.

In [30]:
a, b = foundC
print("phản ví dụ:", a, b)
print("sai  (không gán 0):", wrong_no_reset(a, b))
print("đúng              :", Solution().findLength(a, b))

phản ví dụ: [2, 0, 1, 1] [1, 2, 1]
sai  (không gán 0): 2
đúng              : 1


**Vì sao sai.** Giá trị cũ ở `f[j]` bị coi là "đoạn đang kéo dài" ở hàng mới, nên hai đoạn khớp **bị tách bởi một vết lệch** vẫn được nối thành một. **Cách sửa:** thêm `else: f[j] = 0`.

### Lỗi 4: trả `f[m][n]` thay vì `max` cả bảng

`f[i][j]` là đoạn chung **kết thúc đúng tại** `(i, j)`; đoạn chung dài nhất có thể kết thúc ở bất kỳ đâu, không nhất thiết ở góc dưới phải.

In [31]:
a, b = [1, 2, 3], [1, 2, 4]
print("sai  (f[m][n]):", wrong_last_cell(a, b))
print("đúng          :", Solution().findLength(a, b))

sai  (f[m][n]): 0
đúng          : 2


**Vì sao sai.** Đoạn chung `[1, 2]` kết thúc ở ô `(2, 2)`, còn ô cuối `(3, 3)` ứng với `3` vs `4` (lệch) nên bằng `0`. **Cách sửa:** cập nhật `ans = max(ans, f[i][j])` trong vòng lặp.

### Lỗi 5: bảng 2 chiều bị **dùng chung hàng**

`[[0] * (n + 1)] * (m + 1)` tạo `m + 1` tham chiếu tới **cùng một** list.

In [32]:
def length_dp2d_aliased(nums1, nums2):
    m, n = len(nums1), len(nums2)
    f = [[0] * (n + 1)] * (m + 1)                 # SAI: mọi hàng là cùng một list
    best = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if nums1[i - 1] == nums2[j - 1]:
                f[i][j] = f[i - 1][j - 1] + 1
                best = max(best, f[i][j])
    return best

rng = random.Random(5)
foundAlias = None
for _ in range(20000):
    a = [rng.randint(0, 2) for _ in range(rng.randint(2, 5))]
    b = [rng.randint(0, 2) for _ in range(rng.randint(2, 5))]
    if length_dp2d_aliased(a, b) != oracle_sets(a, b):
        foundAlias = (a, b)
        break
a, b = foundAlias
print("phản ví dụ:", a, b)
print("sai  (alias):", length_dp2d_aliased(a, b))
print("đúng        :", Solution().findLength(a, b))
assert length_dp2d_aliased(a, b) != Solution().findLength(a, b)

phản ví dụ: [1, 0, 2, 1] [1, 2, 0, 1]
sai  (alias): 2
đúng        : 1


**Vì sao sai.** (Với một số input bản sai vẫn ra đúng do may mắn, nên ta tìm một phản ví dụ.) Cả `m + 1` hàng là một list nên `f[i - 1][j - 1]` và `f[i][j]` đọc/ghi cùng một nơi; các giá trị bị trộn lẫn giữa các hàng. **Cách sửa:** `[[0] * (n + 1) for _ in range(m + 1)]`.

### Lỗi 6: tìm kiếm nhị phân bắt đầu từ `lo = 1` — sai khi đáp án là `0`

Nếu có thể **không có phần tử chung nào**, đáp án `0` phải nằm trong miền tìm kiếm.

In [33]:
def wrong_binary_lo1(nums1, nums2):
    m, n = len(nums1), len(nums2)

    def check(L):
        win = {tuple(nums1[i:i + L]) for i in range(m - L + 1)}
        return any(tuple(nums2[j:j + L]) in win for j in range(n - L + 1))

    lo, hi = 1, min(m, n)                       # SAI: bỏ sót đáp án 0
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if check(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

print("sai  (lo = 1):", wrong_binary_lo1([1], [2]))       # 1
print("đúng         :", Solution().findLength([1], [2]))  # 0
assert wrong_binary_lo1([1], [2]) != 0

sai  (lo = 1): 1
đúng         : 0


**Vì sao sai.** Vòng lặp luôn trả về một giá trị `≥ 1` mà không bao giờ kiểm tra `P(1)`. **Cách sửa:** `lo = 0` (vì `P(0)` luôn đúng — mảng con rỗng).

### Lỗi 7: băm trùng nhưng **không kiểm chứng** lại

Băm chỉ là bộ lọc nhanh. Với modulo nhỏ, hai mảng con khác nhau rất dễ có cùng số băm; nếu coi "băm trùng" là "bằng nhau" ta sẽ báo có đoạn chung giả.

In [34]:
def has_common_unverified(a, b, L, mod=13, base=3):
    """SAI: chỉ so số băm (mod nhỏ), không so lại mảng con thật."""
    def hashes(arr):
        out = set()
        for i in range(len(arr) - L + 1):
            h = 0
            for x in arr[i:i + L]:
                h = (h * base + x + 1) % mod
            out.add(h)
        return out
    return bool(hashes(a) & hashes(b))

rng = random.Random(77)
bad = None
for _ in range(20000):
    a = [rng.randint(0, 100) for _ in range(rng.randint(4, 8))]
    b = [rng.randint(0, 100) for _ in range(rng.randint(4, 8))]
    L = 3
    truth = any(tuple(a[i:i + L]) in {tuple(b[j:j + L]) for j in range(len(b) - L + 1)} for i in range(len(a) - L + 1))
    if has_common_unverified(a, b, L) != truth:
        bad = (a, b, L, truth)
        break
print("phản ví dụ va chạm:", bad)
a, b, L, truth = bad
print("băm (mod = 13) báo có đoạn chung độ dài", L, ":", has_common_unverified(a, b, L), "| thực tế:", truth)
assert bad is not None

phản ví dụ va chạm: ([41, 25, 30, 24, 14, 37], [71, 78, 30, 18, 71, 80, 84], 3, False)
băm (mod = 13) báo có đoạn chung độ dài 3 : True | thực tế: False


**Vì sao sai.** Mô-đun `13` chỉ có `13` giá trị băm nên rất nhiều cặp khác nhau trùng. **Cách sửa:** (a) chọn mô-đun rất lớn (như `2^61 - 1`), **và** (b) khi băm trùng, **so lại thật** `nums1[i:i+L] == nums2[j:j+L]` như ở cách 4. Với `(b)` thì kết quả luôn đúng bất kể va chạm (chỉ có thể chậm hơn).

## 14. Mẹo & nhận dạng pattern

- **"Mảng con / chuỗi con liên tiếp chung dài nhất của hai chuỗi"** → DP **kết thúc tại** `(i, j)` với `f[i][j] = f[i-1][j-1] + 1` nếu khớp, ngược lại `0`; đáp án là **max** bảng. (Họ hàng: *Longest Common Substring*; **khác** *Longest Common Subsequence (1143)* ở chỗ lệch thì `max(f[i-1][j], f[i][j-1])`.)
- **Hỏi liên tiếp → "đứt thì về 0"; hỏi dãy con → "đứt thì giữ lại max".** Hãy tự hỏi câu này khi thấy bài hai chuỗi.
- **Tối ưu bộ nhớ DP hai chiều**: nếu hàng `i` chỉ phụ thuộc hàng `i - 1` và ô chéo thì duyệt **ngược** (phải sang trái) trên một mảng, hoặc dùng hai mảng `prev`/`cur`.
- **Tìm kiếm nhị phân trên đáp án** khi điều kiện đơn điệu; kết hợp **băm cuộn** để kiểm tra nhanh, nhưng luôn **kiểm chứng lại** khi băm trùng.
- **Đường chéo**: các bài hai chuỗi/hai mảng có thể nhìn là duyệt các độ lệch `d`; mỗi đường chéo là một bài "đoạn liên tiếp dài nhất" độc lập.
- **Ràng buộc `1000 × 1000`** gợi ý `O(m · n)` là đủ; `O(m · n · min)` không đủ.
- **Phần tử nhỏ (`0..100`)**: cho phép mã hóa mỗi phần tử thành một ký tự để dùng cấu trúc chuỗi (suffix automaton/suffix array) nếu cần `O(m + n)`.

## 15. Biến thể

### 15.1. Trả về **chính mảng con** (không chỉ độ dài)

Giữ lại vị trí kết thúc của đoạn dài nhất trong bảng DP.

In [35]:
def longest_common_subarray(nums1, nums2):
    m, n = len(nums1), len(nums2)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    best, end_i = 0, 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if nums1[i - 1] == nums2[j - 1]:
                f[i][j] = f[i - 1][j - 1] + 1
                if f[i][j] > best:
                    best, end_i = f[i][j], i
    return nums1[end_i - best:end_i]


print(longest_common_subarray([1, 2, 3, 2, 1], [3, 2, 1, 4, 7]))     # [3, 2, 1]
assert longest_common_subarray([1, 2, 3, 2, 1], [3, 2, 1, 4, 7]) == [3, 2, 1]
assert longest_common_subarray([1], [2]) == []
rng = random.Random(15)
for _ in range(200):
    a, b = random_pair(rng, 2, rng.randint(1, 10), rng.randint(1, 10), False)
    seg = longest_common_subarray(a, b)
    assert len(seg) == oracle_sets(a, b)

[3, 2, 1]


### 15.2. Dãy con chung dài nhất (LeetCode 1143)

Chỉ đổi nhánh "lệch": giữ lại `max` của hai ô lân cận.

In [36]:
def lcs_length(a, b):
    m, n = len(a), len(b)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                f[i][j] = f[i - 1][j - 1] + 1
            else:
                f[i][j] = max(f[i - 1][j], f[i][j - 1])
    return f[m][n]

print(lcs_length(list("abcde"), list("ace")))          # 3
assert lcs_length(list("abcde"), list("ace")) == 3
assert lcs_length([1, 2, 3], [1, 3]) == 2                # dãy con chung dài 2 nhưng mảng con chung chỉ 1
assert Solution().findLength([1, 2, 3], [1, 3]) == 1

3


### 15.3. Chuỗi ký tự (Longest Common Substring)

Cùng thuật toán, đổi `list` số thành `str` (chuỗi cũng truy cập `s[i]` được).

In [37]:
def longest_common_substring(s, t):
    return Solution().findLength(list(map(ord, s)), list(map(ord, t)))

print(longest_common_substring("abcdxyz", "xyzabcd"))      # 4 ("abcd")
assert longest_common_substring("abcdxyz", "xyzabcd") == 4
assert longest_common_substring("abc", "def") == 0

4


### 15.4. Mảng con chung của **nhiều** mảng

Tìm kiếm nhị phân trên `L` vẫn dùng được: `check(L)` = giao của các tập cửa sổ độ dài `L` của mọi mảng có khác rỗng không.

In [38]:
def longest_common_subarray_many(arrays):
    def check(L):
        common = None
        for arr in arrays:
            wins = {tuple(arr[i:i + L]) for i in range(len(arr) - L + 1)}
            common = wins if common is None else common & wins
            if not common:
                return False
        return True

    lo, hi = 0, min(len(a) for a in arrays)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if check(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

print(longest_common_subarray_many([[1, 2, 3, 4], [9, 2, 3, 4, 5], [2, 3, 4, 7]]))      # 3 ([2,3,4])
assert longest_common_subarray_many([[1, 2, 3, 4], [9, 2, 3, 4, 5], [2, 3, 4, 7]]) == 3
assert longest_common_subarray_many([[1, 2], [3, 4]]) == 0
rng = random.Random(154)
for _ in range(200):
    a, b = random_pair(rng, 3, rng.randint(1, 10), rng.randint(1, 10), rng.random() < 0.5)
    assert longest_common_subarray_many([a, b]) == oracle_sets(a, b)

3


### 15.5. Biến thể để tự nghĩ

- **Đếm số cặp vị trí `(i, j)`** có mảng con chung độ dài đúng `L`: đếm số ô `f[i][j] ≥ L` (xem bài tập 5).
- **Cho phép sai khác `k` phần tử** (*Longest Common Subarray with at most k mismatches*): cửa sổ trượt trên từng đường chéo.
- **Ba mảng**: DP 3 chiều `O(m · n · p)` hoặc nhị phân + giao tập hợp như trên.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn mọi cặp bắt đầu | mở rộng từ từng cặp `(i, j)` | O(m · n · min(m, n)) | O(1) | chỉ để làm oracle/hiểu đề |
| 2. DP 2 chiều | `f[i][j] = f[i-1][j-1] + 1` hoặc `0` | O(m · n) | O(m · n) | học DP, cần in/truy vết đoạn |
| 3. DP một hàng | duyệt `j` ngược, gán `0` khi lệch | O(m · n) | O(n) | **cách nộp mặc định** |
| 4. Nhị phân độ dài + băm | `P(L)` đơn điệu, kiểm tra bằng băm cuộn | O((m + n) log min) | O(m + n) | khi `m, n` lớn (hàng chục nghìn trở lên) |
| 5. Trượt đường chéo | mỗi độ lệch `d`, đếm `run` khớp liên tiếp | O(m · n) | O(1) | không muốn dùng bảng; dễ mở rộng "sai tối đa `k`" |

**Lời khuyên.** Nắm cách 2 và 3 (công thức "khớp thì chéo + 1, lệch thì 0"), hiểu cách 5 như góc nhìn đường chéo, và biết cách 4 như hướng tiệm cận tốt hơn khi dữ liệu lớn.

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [39]:
print(Solution().findLength([1, 2, 3, 2, 1], [3, 2, 1, 4, 7]))
print(Solution().findLength([1, 2, 3], [1, 3]))
print(Solution().findLength([7, 7], [7, 7, 7]))

3
1
2


**Bài 2 (tính tay).** Với `nums1 = [1, 2, 3]` và `nums2 = [2, 3, 4]`, lập bảng `f` (ba hàng ứng với `1, 2, 3`, ba cột ứng với `2, 3, 4`, **không** tính hàng/cột `0`) theo công thức mục 6 và gán vào `ans2` (đang là `None`) dạng list các list.

In [40]:
ans2 = None      # TODO: [[...], [...], [...]]

**Bài 3 (điền TODO).** Viết `run_length(a, b, i, j)` trả về độ dài đoạn khớp **bắt đầu tại** `a[i]` và `b[j]` (mở rộng chừng nào còn bằng nhau và trong biên) — chính là phần mở rộng của cách 1. Ví dụ `run_length([1,2,3], [0,1,2,4], 0, 1) == 2`.

In [41]:
def run_length(a, b, i, j):
    # TODO: while ... so a[i + k] với b[j + k]
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_find_length(nums1, nums2)` bằng **DP một hàng** (cách 3), nhớ duyệt `j` ngược và gán `0` khi lệch.

In [42]:
def my_find_length(nums1, nums2):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) `count_common_windows(nums1, nums2, L)`: đếm số cặp `(i, j)` sao cho `nums1[i:i+L] == nums2[j:j+L]` (với `L ≥ 1`). Gợi ý: dùng bảng DP và đếm ô có `f ≥ L`. (b) `longest_common_subarray_k_mismatch(nums1, nums2, k)`: độ dài lớn nhất của hai mảng con **cùng độ dài** của `nums1` và `nums2` khác nhau **tối đa `k`** vị trí (so từng vị trí tương ứng). Gợi ý: trên mỗi đường chéo, cửa sổ trượt có tối đa `k` vị trí lệch.

In [43]:
def count_common_windows(nums1, nums2, L):
    # TODO
    pass

def longest_common_subarray_k_mismatch(nums1, nums2, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: ví dụ 1 là `[3,2,1]`; ví dụ 2: dãy con chung `[1,3]` không liên tiếp; ví dụ 3: hai mảng toàn `7`.
- Bài 2: ô `(i, j)` bằng `0` nếu phần tử lệch.
- Bài 3: `k = 0`, tăng `k` khi bằng nhau.
- Bài 4: nhớ hai chi tiết: duyệt `j` ngược, gán `0` khi lệch.
- Bài 5: (a) đoạn dài `≥ L` kết thúc ở ô đó thì có đúng một cặp cửa sổ độ dài `L` kết thúc tại ô đó; (b) cửa sổ trượt trên đường chéo.

**Mức 2 (vừa):**
- Bài 2: hàng của `1`: không khớp gì; hàng của `2`: khớp cột `2` (đầu tiên); hàng của `3`: khớp cột `3` và có chéo `= 1`.
- Bài 3: `while i + k < len(a) and j + k < len(b) and a[i + k] == b[j + k]: k += 1`.
- Bài 4: `for x in nums1: for j in range(n, 0, -1): f[j] = f[j - 1] + 1 if x == nums2[j - 1] else 0`.
- Bài 5: (a) tạo bảng như cách 2 và `sum(1 for i, j if f[i][j] >= L)`; (b) với mỗi độ lệch `d`, hai con trỏ `l`, `r` và đếm `bad` (số vị trí lệch trong cửa sổ); khi `bad > k` thì thu hẹp `l`.

**Mức 3 (gần đáp án):**
- Bài 2: `[[0, 0, 0], [1, 0, 0], [0, 2, 0]]`.
- Bài 4: xem mục 9, bỏ chú thích.
- Bài 5: (a) vì `f[i][j] ≥ L` ⇔ `nums1[i-L:i] == nums2[j-L:j]`; (b) cửa sổ trượt: cập nhật `best = max(best, r - l + 1)` sau khi đảm bảo `bad <= k`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [44]:
# Bài 1
assert Solution().findLength([1, 2, 3, 2, 1], [3, 2, 1, 4, 7]) == 3
assert Solution().findLength([1, 2, 3], [1, 3]) == 1
assert Solution().findLength([7, 7], [7, 7, 7]) == 2

# Bài 2: f theo hàng 1,2,3 và cột 2,3,4
ans2 = [[0, 0, 0], [1, 0, 0], [0, 2, 0]]
a, b = [1, 2, 3], [2, 3, 4]
f = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
for i in range(1, len(a) + 1):
    for j in range(1, len(b) + 1):
        if a[i - 1] == b[j - 1]:
            f[i][j] = f[i - 1][j - 1] + 1
assert ans2 == [row[1:] for row in f[1:]]
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [45]:
# Bài 3
def run_length(a, b, i, j):
    k = 0
    while i + k < len(a) and j + k < len(b) and a[i + k] == b[j + k]:
        k += 1
    return k

assert run_length([1, 2, 3], [0, 1, 2, 4], 0, 1) == 2
assert run_length([1, 2, 3], [1, 2, 3], 0, 0) == 3
assert run_length([1, 2, 3], [4, 5, 6], 0, 0) == 0
assert run_length([1, 2], [1, 2, 3], 1, 1) == 1
# length_brute = max của run_length trên mọi cặp
rng = random.Random(3)
for _ in range(200):
    a, b = random_pair(rng, 2, rng.randint(1, 8), rng.randint(1, 8), False)
    assert max(run_length(a, b, i, j) for i in range(len(a)) for j in range(len(b))) == oracle_sets(a, b)
print("Bài 3 OK")

Bài 3 OK


In [46]:
# Bài 4
def my_find_length(nums1, nums2):
    n = len(nums2)
    f = [0] * (n + 1)
    best = 0
    for x in nums1:
        for j in range(n, 0, -1):
            f[j] = f[j - 1] + 1 if x == nums2[j - 1] else 0
            best = max(best, f[j])
    return best

rng = random.Random(4)
for _ in range(300):
    a, b = random_pair(rng, rng.choice([1, 2, 100]), rng.randint(1, 12), rng.randint(1, 12), rng.random() < 0.5)
    assert my_find_length(a, b) == oracle_sets(a, b)
print("Bài 4 OK")

Bài 4 OK


In [47]:
# Bài 5 (a)
def count_common_windows(nums1, nums2, L):
    m, n = len(nums1), len(nums2)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    total = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if nums1[i - 1] == nums2[j - 1]:
                f[i][j] = f[i - 1][j - 1] + 1
                if f[i][j] >= L:
                    total += 1
    return total

def count_windows_oracle(nums1, nums2, L):
    return sum(1 for i in range(len(nums1) - L + 1) for j in range(len(nums2) - L + 1)
               if nums1[i:i + L] == nums2[j:j + L])

assert count_common_windows([1, 2, 1, 2], [1, 2], 2) == 2
assert count_common_windows([0, 0, 0], [0, 0], 1) == 6
assert count_common_windows([1, 2, 3], [4, 5, 6], 1) == 0
rng = random.Random(51)
for _ in range(300):
    a, b = random_pair(rng, 2, rng.randint(1, 9), rng.randint(1, 9), False)
    L = rng.randint(1, 4)
    assert count_common_windows(a, b, L) == count_windows_oracle(a, b, L)

# Bài 5 (b)
def longest_common_subarray_k_mismatch(nums1, nums2, k):
    m, n = len(nums1), len(nums2)
    best = 0
    for d in range(-(m - 1), n):
        lo, hi = max(0, -d), min(m, n - d)
        l, bad = lo, 0
        for r in range(lo, hi):
            if nums1[r] != nums2[r + d]:
                bad += 1
            while bad > k:
                if nums1[l] != nums2[l + d]:
                    bad -= 1
                l += 1
            best = max(best, r - l + 1)
    return best

def k_mismatch_oracle(nums1, nums2, k):
    best = 0
    for i in range(len(nums1)):
        for j in range(len(nums2)):
            for L in range(1, min(len(nums1) - i, len(nums2) - j) + 1):
                diff = sum(1 for t in range(L) if nums1[i + t] != nums2[j + t])
                if diff <= k:
                    best = max(best, L)
    return best

assert longest_common_subarray_k_mismatch([1, 2, 3], [1, 9, 3], 0) == 1
assert longest_common_subarray_k_mismatch([1, 2, 3], [1, 9, 3], 1) == 3
for _ in range(300):
    a, b = random_pair(rng, 2, rng.randint(1, 8), rng.randint(1, 8), False)
    k = rng.randint(0, 3)
    assert longest_common_subarray_k_mismatch(a, b, k) == k_mismatch_oracle(a, b, k), (a, b, k)
    assert longest_common_subarray_k_mismatch(a, b, 0) == oracle_sets(a, b)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (DP một hàng):**

```python
class Solution:
    def findLength(self, nums1: List[int], nums2: List[int]) -> int:
        n = len(nums2)
        f = [0] * (n + 1)                 # f[j]: đoạn chung dài nhất KẾT THÚC tại nums2[j-1]
        ans = 0
        for x in nums1:
            for j in range(n, 0, -1):     # duyệt NGƯỢC để f[j-1] còn là hàng trước
                if x == nums2[j - 1]:
                    f[j] = f[j - 1] + 1
                    ans = max(ans, f[j])
                else:
                    f[j] = 0              # đứt -> về 0
        return ans
```

**Template (DP 2 chiều):**

```python
f = [[0] * (n + 1) for _ in range(m + 1)]
for i in range(1, m + 1):
    for j in range(1, n + 1):
        if nums1[i - 1] == nums2[j - 1]:
            f[i][j] = f[i - 1][j - 1] + 1
            ans = max(ans, f[i][j])
```

**Câu thần chú:** *mảng con liên tiếp: khớp thì chéo + 1, lệch thì về 0, đáp án là max cả bảng (không phải ô cuối); dãy con (LCS) mới là max hai ô lân cận.*

**Checklist trước khi submit:**

- [ ] Khi hai phần tử **lệch** thì đặt `f = 0` (không phải `max` của lân cận)?
- [ ] Đáp án là **max** của mọi `f`, không phải `f[m][n]`?
- [ ] Bản một hàng: duyệt `j` **ngược** và có nhánh `else: f[j] = 0`?
- [ ] Bảng 2 chiều tạo bằng `[[0]*(n+1) for _ in range(m+1)]` (không dùng `*` ngoài)?
- [ ] Nếu dùng nhị phân: `lo = 0` và `check` đơn điệu; nếu dùng băm: có so lại thật khi băm trùng?
- [ ] Đã thử: mảng một phần tử, không phần tử chung, toàn `0` (đáp án `min(m, n)`), một mảng là mảng con của mảng kia, ví dụ của đề?